In [63]:
import pandas as pd

In [64]:
data = pd.read_csv(r'../Data/toyota_valuation.csv')
data

,make,model,trim,bodyType,regionalSpec,cylinders,color,origin,weightInKg,noOfPassengers
0,TOYOTA,COROLLA,SE,SEDAN,GCC,4.0,SILVER,TAIWAN,1000.0,5.0
1,TOYOTA,HIACE,GLX,MINI BUS,GCC,4.0,WHITE,JAPAN,2000.0,2.0
2,TOYOTA,HIACE,DX,MINI BUS,GCC,4.0,WHITE,JAPAN,1837.0,15.0
3,TOYOTA,HILUX,DL,PICKUP DOUBLE CAB,GCC,4.0,WHITE,JAPAN,1837.0,5.0
4,TOYOTA,CAMRY,XLI,SEDAN,NON-GCC,4.0,BROWN AND BEIGE,USA,1481.0,5.0
...,...,...,...,...,...,...,...,...,...,...
239209,TOYOTA,HILUX,GLX,PICKUP,GCC,6.0,WHITE/PEARLS,INDIA,NaN,5.0
239210,TOYOTA,LAND CRUISER,NaN,SUV,GCC,6.0,BLACK,JAPAN,NaN,7.0
239211,TOYOTA,LAND CRUISER,GXR,SUV,GCC,8.0,WHITE/PEARLS,JAPAN,NaN,8.0
239212,TOYOTA,LAND CRUISER,GXL,SUV,GCC,4.0,WHITE/PEARLS,JAPAN,NaN,7.0


In [65]:
print(data['model'].unique())

['COROLLA' 'HIACE' 'HILUX' 'CAMRY' 'LAND CRUISER' 'YARIS' 'FJ CRUISER'
 'RAV4' 'TUNDRA' 'AVALON' 'FORTUNER' 'COASTER' 'INNOVA' 'SIENNA' 'TACOMA'
 'RUSH' 'PRIUS' 'AVANZA' 'SEQUOIA' 'SUPRA' 'AURION' 'ECHO' 'PREVIA' 'BZ4X'
 'C-HR' 'XA' 'GRANVIA' 'VELOZ' 'RAIZE' 'IQ' 'CROWN' '86' '4RUNNER'
 'URBAN CRUISER' 'SCION' 'HIGHLANDER' nan 'DYNA' 'TERCEL' 'LITEACE'
 'MATRIX' 'ZELAS' 'ALPHARD' 'CRESSIDA' 'BZ3X' 'RUMION' 'MR2' 'WISH'
 'VENZA' 'CELICA' 'T100' 'ALLION' 'VIOS' 'CORONA' 'WIGO' 'DYNA VAN CARGO'
 'MIRAI']


In [66]:
print(data['model'].nunique())

56


In [67]:
print(data['trim'].unique())

['SE' 'GLX' 'DX' 'DL' 'XLI' 'VX' 'MID' 'DLX' 'VXR' 'FABRIC' 'E' 'TOP'
 'EXTREME' 'GL' 'BASE' 'S' 'FLEET' 'STD' 'GXR' 'EXR' 'GL2' 'SR5' 'LE'
 'LE (NON GCC)' 'GLI' 'TX' 'LIMITED' 'XLE (NON GCC)' 'EX' 'ICONIC'
 'VX MID' 'LE HEV' 'SGLX' 'PW CL' 'GX' 'XLE' 'SPORT' 'G' 'GRANDE' 'GR'
 'S TRD SPORT PK' 'S+' 'PRO' 'GLE' 'GLS' 'PLATINUM' 'DLX TOP' 'TURBO G RA'
 'VXR HEV' 'GLE HEV' 'SAFARI' 'LX' 'CE (NON GCC)' 'GLI HEV' 'PREMIUM' 'XL'
 'LX AC PWR OPT' 'LX WINCH' 'TX-L' 'XLE TOURING' 'DLS' 'OTHER'
 'GLE-X HEV 40TH ANNIV' nan 'ADVENTURE' 'CREWMAX CAPSTONE (NON GCC)'
 'LIMITED MID' 'TX-L TOP' 'EXCLUSIVE' 'XSE (NON GCC)' 'BLACK EDITION'
 'DX H/R W/B' 'C (NON GCC)' 'GLX SPORT Z' 'S PLUS' 'VTX' 'GXL' 'HT SWB'
 'ECO' 'GX MID' 'LWB HIGH ROOF' '1794 EDITION (NON GCC)'
 '40TH ANNIVERSARY EDITION' 'TOURING' 'HEV' 'PS PW CL' 'GR SPORT' 'LWB'
 'EXR HEV' 'BE' 'GT' 'SWB' 'SR' 'GTX' 'VX2' 'STANDARD' 'VX1' 'DYNAMIC'
 'FJ STD' 'YX' 'STD AC' 'GLE HEV BLACK EDITION' 'Y PLUS' 'XTREME'
 'HARD TOP' 'HYBRID (NON GCC)' '

In [68]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import numpy as np

# 1. Load the 'year' column from aligned toyota.csv
print("Loading year column from aligned toyota.csv...")
try:
    toyota_orig = pd.read_csv('../Data/toyota.csv')
    data['year'] = toyota_orig['year']
except Exception as e:
    print(f"Error loading year from toyota.csv: {e}")
    try:
        data['year'] = pd.read_csv('../Data/toyota_cleaned.csv')['year']
    except Exception as e2:
        print(f"Error loading year from toyota_cleaned.csv: {e2}")

# Fill missing years with 2026.0 (baseline for new cars)
data['year'] = pd.to_numeric(data['year'], errors='coerce').fillna(2026.0)

# 2. Scrape base MSRP from the official website
scraped_cache = {}

def get_toyota_msrp(model):
    if pd.isna(model):
        return None
    
    model_name = str(model).upper().strip()
    if model_name in scraped_cache:
        return scraped_cache[model_name]
    
    # Mapping from database model name to URL slug
    slug_mapping = {
        "LAND CRUISER": "land-cruiser",
        "LAND CRUISER PRADO": "prado",
        "PRADO": "prado",
        "URBAN CRUISER": "urban-cruiser",
        "C-HR": "c-hr",
        "BZ4X": "bz4x",
        "GRANVIA": "granvia",
        "RAIZE": "raize",
        "VELOZ": "veloz",
        "HIACE": "hiace",
        "COASTER": "coaster",
        "LITEACE": "liteace",
        "INNOVA": "innova",
    }
    
    model_slug = slug_mapping.get(model_name, model_name.lower().replace(" ", "-"))
    url = f"https://www.toyota.ae/en/new-cars/{model_slug}/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            
            # Check PriceRange class first
            price_range_tag = soup.find(class_=re.compile("(?i)PriceRange"))
            if price_range_tag:
                text = price_range_tag.get_text()
                match = re.search(r"([\d,]+)", text)
                if match:
                    price_val = int(match.group(1).replace(",", ""))
                    scraped_cache[model_name] = price_val
                    return price_val
            
            # Fallback to PriceDetails or PriceRange tags
            price_details_tags = soup.find_all(class_=re.compile("(?i)(PriceDetails|PriceRange)"))
            for tag in price_details_tags:
                text = tag.get_text()
                numbers = [int(num.replace(",", "")) for num in re.findall(r"\b\d{2,3},\d{3}\b", text)]
                if numbers:
                    price_val = max(numbers)
                    scraped_cache[model_name] = price_val
                    return price_val
    except Exception as e:
        print(f"Failed to scrape for {model}: {e}")
        
    scraped_cache[model_name] = None
    return None

# Fallbacks for all model lines (ensures trusted historical/baseline prices)
base_fallbacks = {
    "COROLLA": 76900,
    "CAMRY": 122900,
    "YARIS": 64900,
    "FORTUNER": 128900,
    "LAND CRUISER": 239900,
    "HILUX": 115900,
    "HIACE": 99900,
    "RAV4": 105900,
    "VELOZ": 81900,
    "URBAN CRUISER": 79900,
    "SUPRA": 238900,
    "INNOVA": 126900,
    "HIGHLANDER": 205900,
    "CROWN": 199900,
    "RAIZE": 63900,
    "RUSH": 71900,
    "COASTER": 218900,
    "GRANVIA": 216900,
    "LITEACE": 69900,
    "PRADO": 199900,
    "LAND CRUISER PRADO": 199900,
    "AVALON": 145000,
    "TUNDRA": 195000,
    "SIENNA": 180000,
    "TACOMA": 150000,
    "PRIUS": 115000,
    "AVANZA": 68000,
    "SEQUOIA": 240000,
    "AURION": 120000,
    "ECHO": 45000,
    "PREVIA": 130000,
    "BZ4X": 189900,
    "C-HR": 95000,
    "XA": 50000,
    "IQ": 55000,
    "86": 162900,
    "4RUNNER": 160000,
    "SCION": 70000,
    "SOLARA": 85000,
    "DYNA": 110000,
    "TERCEL": 40000,
    "MATRIX": 65000,
    "ZELAS": 95000,
    "ALPHARD": 250000,
    "CRESSIDA": 35000,
    "BZ3X": 140000,
    "RUMION": 85000,
    "MR2": 110000,
    "WISH": 80000,
    "VENZA": 135000,
    "CELICA": 90000,
    "T100": 70000,
    "ALLION": 75000,
    "VIOS": 70000,
    "WIGO": 45000,
    "MIRAI": 220000
}

# Define Trim Premiums
TRIM_PREMIUMS = {
    "GLI": 7000,
    "SE": 12000,
    "GLX": 12000,
    "GLE": 7000,
    "G": 9000,
    "LIMITED": 25000,
    "SPORT": 15000,
    "XLE": 20000,
    "XSE": 25000,
    "PLATINUM": 30000,
    "TOP": 18000,
    "GXR": 34000,
    "VXR": 67000,
    "VX": 30000,
    "ZX": 159000,
    "GR": 159000,
    "ADVENTURE": 33000,
    "EXR": 15000,
    "SR5": 20000,
    "TRD": 25000,
    "STD": 0,
    "DX": 0,
    "DLX": 5000,
    "GL": 5000,
}

def calculate_ex_showroom_price(model, trim, year):
    m_name = str(model).upper().strip()
    t_name = str(trim).upper().strip()
    
    base_price = get_toyota_msrp(m_name)
    if base_price is None:
        base_price = base_fallbacks.get(m_name, base_fallbacks.get("COROLLA"))
        
    premium = TRIM_PREMIUMS.get(t_name, 0)
    new_price = base_price + premium
    
    y_val = float(year) if not pd.isna(year) else 2026.0
    age = max(0.0, 2026.0 - y_val)
    
    # GCC annual depreciation (12% per year)
    depreciated_price = new_price * ((1.0 - 0.12) ** age)
    return max(depreciated_price, min(2000.0, new_price * 0.05))

# 3. Create unique vehicles and calculate current_msrp and price
print("Calculating MSRP and prices for unique combinations...")
unique_df = (
    data[["make", "model", "trim", "year"]]
    .fillna({"model": "UNKNOWN", "trim": "UNKNOWN", "year": 2026.0})
    .drop_duplicates()
)

# Apply scraping for base MSRP
unique_df["current_msrp"] = unique_df["model"].apply(get_toyota_msrp)

# Handle fallbacks for current_msrp
unique_df["current_msrp"] = unique_df.apply(
    lambda r: r["current_msrp"] if r["current_msrp"] is not None else base_fallbacks.get(str(r["model"]).upper().strip(), base_fallbacks.get("COROLLA")),
    axis=1
)

# Apply price calculation on the unique dataframe
unique_df["price"] = unique_df.apply(
    lambda r: calculate_ex_showroom_price(r["model"], r["trim"], r["year"]), 
    axis=1
)

# Merge back while preserving all existing columns
print("Merging calculated prices back into the main DataFrame...")
if "current_msrp" in data.columns:
    data = data.drop(columns=["current_msrp"])
if "price" in data.columns:
    data = data.drop(columns=["price"])

data = data.merge(
    unique_df,
    on=["make", "model", "trim", "year"],
    how="left"
)

# Save updated dataset to CSV
output_path = r"../Data/toyota_valuation.csv"
data.to_csv(output_path, index=False)
print(f"Saved updated dataset to {output_path}")

# 4. Group by model, trim, and year, and store as data_new
data_new = data.dropna(subset=["model"]).groupby(["model", "trim", "year"])["price"].first().reset_index()

'\n======================================================================'

'MODEL: 4RUNNER'

,make,model,trim,bodyType,regionalSpec,cylinders,color,origin,weightInKg,noOfPassengers
0,TOYOTA,4RUNNER,LIMITED,SUV,GCC,6.0,GRAY AND BLACK,JAPAN,2100.0,5.0
1,TOYOTA,4RUNNER,LIMITED,SUV,GCC,6.0,WHITE,JAPAN,1600.0,5.0
2,TOYOTA,4RUNNER,LIMITED,SUV,GCC,6.0,BLACK AND RED,JAPAN,1800.0,5.0
3,TOYOTA,4RUNNER,SR5,SUV,GCC,6.0,SILVER,JAPAN,2033.0,5.0
4,TOYOTA,4RUNNER,SR5,SUV,GCC,8.0,WHITE,JAPAN,1800.0,5.0
...,...,...,...,...,...,...,...,...,...,...
265,TOYOTA,4RUNNER,SR5,SUV,GCC,6.0,SILVER,JAPAN,900.0,5.0
266,TOYOTA,4RUNNER,LIMITED,SUV,GCC,6.0,WHITE,JAPAN,1500.0,5.0
267,TOYOTA,4RUNNER,SR5,SUV,NON-GCC,4.0,WHITE,JAPAN,NaN,5.0
268,TOYOTA,4RUNNER,SR5,SUV,NON-GCC,6.0,BLUE,JAPAN,NaN,5.0


'Rows: 270'

'\n======================================================================'

...

'Rows: 23248'

'\n======================================================================'

'MODEL: ZELAS'

,make,model,trim,bodyType,regionalSpec,cylinders,color,origin,weightInKg,noOfPassengers
0,TOYOTA,ZELAS,MID,COUPE,GCC,4.0,WHITE,JAPAN,1080.0,5.0
1,TOYOTA,ZELAS,SPORT,COUPE,GCC,4.0,GREY,JAPAN,1459.0,5.0
2,TOYOTA,ZELAS,MID,COUPE,GCC,4.0,BLACK AND SILVER,JAPAN,1200.0,5.0
3,TOYOTA,ZELAS,SPORT,COUPE,GCC,4.0,GRAY AND BLACK,JAPAN,1499.0,5.0
4,TOYOTA,ZELAS,SPORT,COUPE,GCC,4.0,SILVER,JAPAN,1200.0,5.0
...,...,...,...,...,...,...,...,...,...,...
87,TOYOTA,ZELAS,SPORT,COUPE,GCC,4.0,SILVER,JAPAN,1223.0,5.0
88,TOYOTA,ZELAS,SPORT,COUPE,GCC,4.0,BLACK,JAPAN,NaN,5.0
89,TOYOTA,ZELAS,MID,COUPE,GCC,4.0,MAROON,JAPAN,NaN,5.0
90,TOYOTA,ZELAS,STD,HATCHBACK,GCC,4.0,GRAY,JAPAN,NaN,5.0


'Rows: 92'

In [ ]:
data_new
